# 02 · Feature engineering & leakage checks
Builds the training table and verifies the no-leakage contract.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src import ingest, features as F, config as C
data = ingest.run()
X, y, meta = F.build_training_table(data['historical'], data['teams'])
print('X', X.shape, '| features', len(F.FEATURE_COLUMNS))
X.head()

## Class balance & missing values

In [ ]:
print('class balance:', y.map({0:'home_win',1:'draw',2:'away_win'}).value_counts(normalize=True).round(3).to_dict())
na = X.isna().mean().sort_values(ascending=False)
display(na[na > 0].to_frame('missing_frac'))

## Leakage check
Historical win-rate for a match in year *Y* must use only matches with `year < Y`.
We re-derive the cutoff dicts and confirm a match's own result never feeds its features.

In [ ]:
hist = data['historical']
y0 = sorted(hist['year'].unique())[0]
wr0 = F.winrates_before(hist, y0)
print('first edition', y0, '-> win-rates available:', len(wr0), '(expected 0 — nothing precedes it)')
assert len(wr0) == 0, 'leakage: data from year Y used for year Y'
print('OK: no pre-match history for the first edition, as required.')

## Feature correlations

In [ ]:
num = ['elo_diff','fifa_rank_diff','market_value_ratio','squad_age_diff','form_diff',
       'home_historical_win_rate','away_historical_win_rate','h2h_home_win_rate','stage_ord']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(X[num].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
plt.title('Numeric feature correlations'); plt.tight_layout(); plt.show()

In [ ]:
Xv, yv = F.build_md1_validation(data['teams'], hist, data['md1'])
print('MD1 validation table:', Xv.shape, '| labels:', yv.value_counts().to_dict())
Xv.head()